In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import random
import pickle

from sklearn.cluster import SpectralClustering

from torchvision.datasets import MNIST, KMNIST, FashionMNIST

from dataset_builder import *
from clustering_utils import *
from Siamese_networks import *
from training_functions import *

In [2]:
def get_semi_supervised_split(dataset_name='MNIST', n=1000, train=True, root='./data'):
    """
    dataset_name: 'MNIST' o 'FashionMNIST'
    n: numero totale di esempi etichettati da restituire (distribuiti equamente sulle classi)
    train: True per train set, False per test
    root: directory di download

    Restituisce:
    - labeled_set: torch.utils.data.Subset con esempi etichettati
    - unlabeled_set: Dataset custom con solo immagini (senza etichette)
    """
    transform = transforms.ToTensor()

    # Carica il dataset
    if dataset_name == 'MNIST':
        full_dataset = MNIST(root=root, train=train, download=True, transform=transform)
    elif dataset_name == 'FashionMNIST':
        full_dataset = FashionMNIST(root=root, train=train, download=True, transform=transform)
    else:
        raise ValueError("dataset_name deve essere 'MNIST' o 'FashionMNIST'")

    targets = np.array(full_dataset.targets)
    labeled_indices = []
    n_per_class = n // 10  # supponiamo 10 classi

    # Estrai n_per_class per ciascuna classe
    for class_label in range(10):
        class_indices = np.where(targets == class_label)[0]
        selected = np.random.choice(class_indices, n_per_class, replace=False)
        labeled_indices.extend(selected)

    labeled_set = Subset(full_dataset, labeled_indices)

    # Unlabeled = tutto il resto
    all_indices = set(range(len(full_dataset)))
    unlabeled_indices = list(all_indices - set(labeled_indices))
    unlabeled_set = Subset(full_dataset, unlabeled_indices)

    return labeled_set, unlabeled_set

In [3]:
def m_unlabeled_samples(m, unlabeled_set):
    """
    Restituisce un sottoinsieme di m campioni non etichettati dal dataset unlabeled_set.
    """
    if m > len(unlabeled_set):
        raise ValueError("m non può essere maggiore della dimensione del dataset unlabeled_set")

    indices = np.random.choice(len(unlabeled_set), m, replace=False)
    return Subset(unlabeled_set, indices)

    